# Qwen 2.5 1.5B Reasoning SFT Warmup — Modal A100 40GB
**Fine-tunes Qwen2.5-1.5B-Instruct on preprocessed reasoning cold-start data**

Single A100 40GB setup on Modal with Unsloth, Flash Attention 2, BF16 native support, HF step weight checkpointing, and lossless 16-bit model merging.

| Feature | Old Kaggle/Colab setup | Modal A100 40GB (this nb) |
|---|---|---|
| Platform | Kaggle / Colab T4 | Modal A100 (40GB VRAM) |
| Data Source | On-the-fly cleaning | Loaded from `abhinav0231/reasoning-cold-start-sft-data` |
| Hardware Precision | FP16 | Native BFloat16 (`torch.bfloat16`) |
| Flash Attention | No / Basic | Flash Attention 2 auto-enabled (cc=8.0) |
| Per-device Batch Size | 1 | 8 (8x higher per-GPU batch capacity) |
| Effective Batch Size | 1 x 4 = 4 | 8 x 2 = 16 |
| HF Checkpointing | No step push | Intermediate step checkpoints pushed to HF |
| Merge & Push | Adapter only | Full 16-bit merged model pushed to HF (`-merged`) for GRPO |

## Cell 1 — Install Dependencies (Modal %uv optimized)

In [ ]:
%uv pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
%uv pip install wandb flash-attn liger-kernel datasets huggingface_hub -q

## Cell 2 — Hardware Probe & Unsloth Verification

In [ ]:
import unsloth  # Must be first import for patches
print(f"Unsloth Version : {unsloth.__version__}")

import torch, trl
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Version    : {torch.version.cuda}")

p = torch.cuda.get_device_properties(0)
print(f"\nGPU Device     : {p.name}")
print(f"VRAM Capacity  : {p.total_memory / 1e9:.1f} GB")
print(f"Compute        : cc={p.major}.{p.minor}")
print(f"BFloat16       : {'Supported' if p.major >= 8 else 'NOT supported'}")
print(f"FlashAttention2: {'Supported' if p.major >= 8 else 'NOT supported'}")

assert torch.cuda.is_available(), "No GPU detected!"
assert p.major >= 8, f"Requires Ampere+ GPU (A100/H100/RTX3090+). Got cc={p.major}.{p.minor}"
print("\n✅ Hardware check passed for A100")

## Cell 3 — Configuration

In [ ]:
import os, torch

HF_USERNAME = "abhinav0231"

# Model & LoRA Parameters
MODEL_NAME     = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LENGTH = 3072
LORA_RANK      = 32
LORA_ALPHA     = 32

# Dataset Repo on HF
DATASET_REPO   = f"{HF_USERNAME}/reasoning-cold-start-sft-data"

# Training Hyperparameters (Optimized for Modal A100 40GB)
NUM_TRAIN_EPOCHS = 2
LEARNING_RATE    = 1e-5
WARMUP_RATIO     = 0.05
BATCH_SIZE       = 8       # Increased for A100 40GB VRAM
GRAD_ACCUM       = 2       # Effective batch size = 8 * 2 = 16
SEED             = 42

# Output Repositories on Hugging Face Hub
HF_ADAPTER_REPO  = f"{HF_USERNAME}/Qwen2.5-1.5B-reasoning-warmup"
HF_MERGED_REPO   = f"{HF_USERNAME}/Qwen2.5-1.5B-reasoning-warmup-merged"
CHECKPOINT_REPO  = f"{HF_USERNAME}/Qwen2.5-1.5B-reasoning-warmup-checkpoints"
SAVE_STEPS       = 100

# Paths on Modal
OUTPUT_DIR = "/root/sft_warmup_output"
MERGED_DIR = "/root/sft_warmup_merged"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MERGED_DIR, exist_ok=True)

# W&B Config
WANDB_PROJECT  = "superqwen"
WANDB_RUN_NAME = "qwen2.5-1.5b-sft-warmup-a100"

DTYPE = torch.bfloat16
print(f"Model           : {MODEL_NAME}")
print(f"Dataset         : {DATASET_REPO}")
print(f"Adapter Output  : {HF_ADAPTER_REPO}")
print(f"Merged Output   : {HF_MERGED_REPO}")
print(f"Checkpoint Repo : {CHECKPOINT_REPO}")
print(f"Batch Config    : Batch={BATCH_SIZE}, GradAccum={GRAD_ACCUM} => EffBatch={BATCH_SIZE*GRAD_ACCUM}")

## Cell 4 — Authentication (Hugging Face & W&B)

In [ ]:
import wandb
from huggingface_hub import login

HF_TOKEN    = os.environ.get("HF_TOKEN", "YOUR_HF_TOKEN_HERE")
WANDB_TOKEN = os.environ.get("WANDB_API_KEY", "YOUR_WANDB_KEY_HERE")

login(token=HF_TOKEN)
wandb.login(key=WANDB_TOKEN, relogin=True)
os.environ["WANDB_API_KEY"] = WANDB_TOKEN
print("✅ Authenticated with HuggingFace & WandB")

## Cell 5 — Load Unsloth Model & Setup LoRA

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = DTYPE,
    load_in_4bit   = True,
)

tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,
    lora_alpha = LORA_ALPHA,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable/1e6:.1f}M / {total/1e6:.0f}M ({100*trainable/total:.1f}%")

## Cell 6 — Load Preprocessed Dataset & Format Chat Template

In [ ]:
from datasets import load_dataset

print(f"Loading preprocessed dataset from HF: {DATASET_REPO} ...")
dataset = load_dataset(DATASET_REPO, split="train")
print(f"✅ Dataset loaded: {len(dataset):,} samples")

def format_sample(ex):
    return {"text": tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False
    )}

dataset = dataset.map(format_sample, batched=False, num_proc=2)
sample = dataset[0]["text"]
assert "<think>" in sample and "<answer>" in sample, "Format check failed!"
print(f"\n--- Sample Preview ---\n{sample[:400]}\n\n✅ Dataset format validated")

## Cell 7 — Intermediate Weight Checkpoint Push Callback

In [ ]:
from transformers import TrainerCallback
from huggingface_hub import upload_folder, HfApi

class CheckpointPushCallback(TrainerCallback):
    """Pushes intermediate LoRA checkpoint adapters to HF Hub after every save step."""
    def __init__(self, repo_id: str, token: str):
        self.repo_id = repo_id
        self.token   = token
        if repo_id:
            try:
                HfApi().create_repo(repo_id, token=token, exist_ok=True, private=True)
                print(f"Checkpoint HF repo initialized: {repo_id}")
            except Exception as e:
                print(f"WARNING: Could not initialize checkpoint repo: {e}")

    def on_save(self, args, state, control, **kwargs):
        if not self.repo_id:
            return
        step = state.global_step
        ckpt_dir = os.path.join(args.output_dir, f"checkpoint-{step}")
        if not os.path.exists(ckpt_dir):
            return
        print(f"\n>> Step {step}: uploading checkpoint to HF Hub -> {self.repo_id} ...")
        try:
            upload_folder(
                folder_path     = ckpt_dir,
                repo_id         = self.repo_id,
                token           = self.token,
                path_in_repo    = f"checkpoint-{step}",
                commit_message  = f"SFT Warmup checkpoint step {step}",
                ignore_patterns = ["*.lock"],
            )
            print(f">> Step {step} checkpoint uploaded successfully.")
        except Exception as e:
            print(f"WARNING: HF checkpoint upload failed at step {step}: {e}")

_callbacks = [CheckpointPushCallback(repo_id=CHECKPOINT_REPO, token=HF_TOKEN)]
print("✅ CheckpointPushCallback configured")

## Cell 8 — Initialize SFTTrainer & Train

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bf16_supported

wandb.init(
    project = WANDB_PROJECT,
    name    = WANDB_RUN_NAME,
    config  = {
        "model": MODEL_NAME, "lora_rank": LORA_RANK,
        "lr": LEARNING_RATE, "epochs": NUM_TRAIN_EPOCHS,
        "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM,
        "samples": len(dataset),
    }
)

training_args = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = NUM_TRAIN_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    lr_scheduler_type           = "cosine",
    warmup_ratio                = WARMUP_RATIO,
    fp16                        = not is_bf16_supported(),
    bf16                        = is_bf16_supported(),
    optim                       = "adamw_torch_fused",
    weight_decay                = 0.01,
    max_grad_norm               = 1.0,
    max_seq_length              = MAX_SEQ_LENGTH,
    dataset_text_field          = "text",
    packing                     = False,
    logging_steps               = 10,
    save_steps                  = SAVE_STEPS,
    save_total_limit            = 2,
    report_to                   = "wandb",
    run_name                    = WANDB_RUN_NAME,
    seed                        = SEED,
    dataloader_num_workers      = 0,
)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset,
    args          = training_args,
    callbacks     = _callbacks,
)

steps = len(dataset) * NUM_TRAIN_EPOCHS // (BATCH_SIZE * GRAD_ACCUM)
print(f"\n{'='*60}\n  SFT Warmup (A100) | {len(dataset)} samples | {steps} steps\n{'='*60}\n")
stats = trainer.train()
print(f"\n✅ Training completed in {stats.metrics['train_runtime']/60:.1f} min")
wandb.finish()

## Cell 9 — Save & Push LoRA Adapter

In [ ]:
print(f"Saving trained LoRA adapter locally to {OUTPUT_DIR}/lora_adapter ...")
model.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")

print(f"Pushing LoRA adapter to Hugging Face Hub: {HF_ADAPTER_REPO} ...")
model.push_to_hub(HF_ADAPTER_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HF_ADAPTER_REPO, token=HF_TOKEN)
print(f"✅ LoRA adapter pushed to https://huggingface.co/{HF_ADAPTER_REPO}")

## Cell 10 — Lossless 16-bit Merge & Push (Required for GRPO)

In [ ]:
from unsloth import FastLanguageModel

print("Loading trained model + adapter in full precision BFloat16 for lossless merge ...")
model_m, tok_m = FastLanguageModel.from_pretrained(
    model_name     = OUTPUT_DIR,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = torch.bfloat16,  # Full precision BF16 merge
    load_in_4bit   = False,           # No quantization artifacts
)

print(f"Merging LoRA weights locally into 16-bit model -> {MERGED_DIR} ...")
model_m.save_pretrained_merged(MERGED_DIR, tok_m, save_method="merged_16bit")
print(f"✅ Merged model saved locally to {MERGED_DIR}")

print(f"Pushing merged 16-bit model to Hugging Face Hub: {HF_MERGED_REPO} ...")
model_m.push_to_hub_merged(HF_MERGED_REPO, tok_m, save_method="merged_16bit", token=HF_TOKEN)
print(f"\n🎉 Successfully merged and pushed 16-bit standalone model to:")
print(f"   https://huggingface.co/{HF_MERGED_REPO}")